# Lesson 3: Downloading float time series and individual profiles ⬇️ 

### 🎯 Learning Objectives
In this lesson, we will learn how to download the float time series and individual profiles identified using the filtered Argo index file (Lesson 1) and figures (Lesson 2).

### 🛠️ Prerequisites
Before starting this lesson, make sure that you have completed **Lesson 2**.
If you are using **Binder**, make sure to upload the relevant index files created in **Lesson 1**.

### ❓ How to Use This Notebook
* 📚 **Read** the tutorial text blocks carefully, as they provide the essential background information behind the code.
* ▶️ **Run** each code cell sequentially by clicking the cell and pressing `Shift + Enter`.
* 📝 **Exercise** your knowledge! At the end of this notebook, we provide active learning exercises where you will need to write or modify the code yourself.

### Ready? Let's Get Started!
---

## 📚 Tutorial

### Import libraries

▶️ **Run** the cell below to import relevant Python libraries for this lesson.

In [1]:
import pandas as pd
import os
import urllib.request

### Time series vs. individual profiles

Synthetic BGC-Argo data are provided as two types: time series and individual profiles. The former is simply a concatenation of the latter, therefore, the former is a 2D dataset (time, depth), whereas the former is a 1D dataset (depth). Below are examples of the two datasets:

* An example of time series: [6903574_Sprof.nc](https://data-argo.ifremer.fr/dac/coriolis/6903574/6903574_Sprof.nc)
* An example of individual profiles: [SD6903574_001.nc](https://data-argo.ifremer.fr/dac/coriolis/6903574/profiles/SD6903574_001.nc)

"Which datasets to use?" depends on your specific application. If you are interested in the the temporal changes, time series may be more suitable. In contrast, if you are interested in spatial distribution, individual profiles may be more suitable.

### Define a function for downloading float time series and individual profiles

▶️ **Run** the cell below to define the function.

In [2]:
def download_argo(df_index, wmoid_input=None):
    """
    Downloads Argo data from active servers. 
    Handles both full Sprof time-series and individual profile .nc files.
    If wmoid_input is left blank, it automatically downloads every file listed in df_index.
    """
    
    # ==========================================
    # 1. Handle Optional Input & Warning
    # ==========================================
    if wmoid_input is None:
        num_files = len(df_index)
        print(f"⚠️ No specific floats provided.")
        print(f"⚠️ Defaulting to download ALL {num_files} files in the provided DataFrame!")
        
        # Extra safety net in case they passed the raw, unfiltered index by mistake!
        if num_files > 1000:
            print("🚨 WARNING: This is a massive download. Press the 'Stop' square in Jupyter if this was a mistake!")
            
        # Automatically grab the 'file' column from the DataFrame
        wmoid_input = df_index['file']

    # ==========================================
    # 2. Test and find a working server
    # ==========================================
    urls_to_try = [
        'https://data-argo.ifremer.fr/',
        'https://usgodae.org/pub/outgoing/argo/'
    ]
    
    active_url = None
    for base_url in urls_to_try:
        try:
            # Send a quick ping to see if the server is awake
            urllib.request.urlopen(base_url, timeout=5)
            active_url = base_url
            print(f"🌍 Successfully connected to {active_url}")
            break
        except Exception:
            print(f"❌ {base_url} is not responding...")

    if not active_url:
        print("🚨 Could not connect to any Argo servers. Please check your internet.")
        return

    # ==========================================
    # 3. Convert input to a standard Python list
    # ==========================================
    if isinstance(wmoid_input, pd.Series):
        # Handles Pandas columns smoothly
        item_list = wmoid_input.tolist()
        
    elif isinstance(wmoid_input, (int, str)):
        # If it's a single integer or string, safely wrap it in a list!
        item_list = [wmoid_input]
        
    else:
        # Assumes it is already a list, tuple, or array
        item_list = list(wmoid_input)

    # ==========================================
    # 4. Loop through the requested items
    # ==========================================
    for item in item_list:
        item_str = str(item)
        
        # ------------------------------------------
        # SCENARIO A: Individual Profile Download
        # Example: 'aoml/1901584/profiles/R1901584_001.nc'
        # ------------------------------------------
        if '.nc' in item_str or '/' in item_str:
            
            # Extract the WMO ID and filename from the long string
            parts = item_str.split('/')
            wmo_str = parts[1]       # e.g., '1901584'
            file_name = parts[-1]    # e.g., 'R1901584_001.nc'
            
            # Setup output directory -> "wmoid/profiles/"
            out_dir = os.path.join(wmo_str, 'profiles')
            os.makedirs(out_dir, exist_ok=True)
            
            # Construct URL and destination
            url = f"{active_url}dac/{item_str}"
            destination = os.path.join(out_dir, file_name)
            
        # ------------------------------------------
        # SCENARIO B: Time Series (Sprof) Download
        # Example: '1901584'
        # ------------------------------------------
        else:
            wmo_str = item_str
            
            # Look up the float in df_index to find its DAC (organization)
            match = df_index[df_index['file'].str.contains(f"/{wmo_str}/", na=False)]
            
            if match.empty:
                print(f"⚠️ WMO ID {wmo_str} not found in the index file. Skipping...")
                continue
                
            file_path = match['file'].values[0]
            orgname = file_path.split('/')[0] # e.g., 'aoml'
            
            # Setup output directory -> "wmoid/"
            out_dir = wmo_str
            os.makedirs(out_dir, exist_ok=True)
            
            # Construct URL and destination
            file_name = f"{wmo_str}_Sprof.nc"
            url = f"{active_url}dac/{orgname}/{wmo_str}/{file_name}"
            destination = os.path.join(out_dir, file_name)

        # ------------------------------------------
        # 5. Perform the actual download
        # ------------------------------------------
        try:
            urllib.request.urlretrieve(url, destination)
            print(f"✅ Download successful: {destination}")
        except Exception as e:
            print(f"❌ Failed to download from {url}\n   Error: {e}")

This function requires the filtered index file (mandatory) and a list of either the list of float  wmo IDs or individual profiles (optional). If the latter list (the second argment) is not provided, it will download all profiles in the filtered index.

### Load the filtered index file

▶️ **Run** the cell below to load the filtered index file created in Lesson 1.

In [3]:
df_index = pd.read_csv('argo_synthetic-profile_index_default.csv',parse_dates=['datetime'])
df_index

,file,latitude,longitude,parameters,datetime,wmo
0,aoml/5906513/profiles/SD5906513_025.nc,27.139,139.571,PRES TEMP PSAL DOXY CHLA CHLA_FLUORESCENCE BBP...,2023-01-04 17:23:25,5906513
1,aoml/5906513/profiles/SD5906513_026.nc,27.193,139.748,PRES TEMP PSAL DOXY CHLA CHLA_FLUORESCENCE BBP...,2023-01-15 02:00:45,5906513
2,aoml/5906513/profiles/SD5906513_027.nc,27.255,139.714,PRES TEMP PSAL DOXY CHLA CHLA_FLUORESCENCE BBP...,2023-01-25 06:42:36,5906513
3,aoml/5906513/profiles/SD5906513_028.nc,27.081,140.029,PRES TEMP PSAL DOXY CHLA CHLA_FLUORESCENCE BBP...,2023-02-04 11:05:10,5906513
4,aoml/5906513/profiles/SD5906513_029.nc,26.829,139.993,PRES TEMP PSAL DOXY CHLA CHLA_FLUORESCENCE BBP...,2023-02-14 15:13:24,5906513
...,...,...,...,...,...,...
313,csio/2902882/profiles/SR2902882_174.nc,20.532,139.938,PRES TEMP PSAL DOXY CHLA BBP532 BBP700 DOWN_IR...,2024-12-11 02:17:22,2902882
314,csio/2902882/profiles/SR2902882_175.nc,20.622,139.968,PRES TEMP PSAL DOXY CHLA BBP532 BBP700 DOWN_IR...,2024-12-16 02:11:21,2902882
315,csio/2902882/profiles/SR2902882_176.nc,20.718,139.975,PRES TEMP PSAL DOXY CHLA BBP532 BBP700 DOWN_IR...,2024-12-21 02:20:16,2902882
316,csio/2902882/profiles/SR2902882_177.nc,20.786,139.957,PRES TEMP PSAL DOXY CHLA BBP532 BBP700 DOWN_IR...,2024-12-26 02:08:23,2902882


### Download the float time series

▶️ **Run** the cell below to download the time series for the float with the WMO ID of 5905613, which is one of the three floats available in this index file (see *map_default.png* in Lesson 2).

In [4]:
download_argo(df_index, 5906513)

🌍 Successfully connected to https://data-argo.ifremer.fr/
✅ Download successful: 5906513/5906513_Sprof.nc


### Download the individual profiles

▶️ **Run** the cell below to download the individual profiles included in the index file **since July 1, 2024** (during which the three floats have consistent profiling cycles; see *timeline_default.png* in Lesson 2).

In [6]:
df_index_limit = df_index[df_index['datetime'] >= pd.to_datetime("2024-07-01")]
download_argo(df_index_limit, wmoid_input=None) # "wmoid_input=None" means download all in the index.

⚠️ No specific floats provided.
⚠️ Defaulting to download ALL 73 files in the provided DataFrame!
🌍 Successfully connected to https://data-argo.ifremer.fr/
✅ Download successful: 5906513/profiles/SD5906513_078.nc
✅ Download successful: 5906513/profiles/SD5906513_079.nc
✅ Download successful: 5906513/profiles/SD5906513_080.nc
✅ Download successful: 5906513/profiles/SD5906513_081.nc
✅ Download successful: 5906513/profiles/SD5906513_082.nc
✅ Download successful: 5906513/profiles/SD5906513_083.nc
✅ Download successful: 5906513/profiles/SD5906513_084.nc
✅ Download successful: 5906513/profiles/SD5906513_085.nc
✅ Download successful: 5906513/profiles/SD5906513_086.nc
✅ Download successful: 5906513/profiles/SD5906513_087.nc
✅ Download successful: 5906513/profiles/SD5906513_088.nc
✅ Download successful: 5906513/profiles/SD5906513_089.nc
✅ Download successful: 5906513/profiles/SD5906513_090.nc
✅ Download successful: 5906513/profiles/SD5906513_091.nc
✅ Download successful: 5906513/profiles/SD5906

**This is the end of the tutorials for Lesson 3. Hope you enjoyed it!**

---

## 📝 Exercises

Using the filtered index files created in Lesson 1 and the figures created in Lesson 2, identify and download the float time series or individual profiles that meet specific criteria using the function `download_argo`.

### Exercise 1: Download the individual profiles of chlorophyll-a in the Southern Ocean on March 1, 2026

From *map_exercise1.png* in Lesson 2, we see the global coverage of chlorophyll-a measurements on March 1, 2026. Suppose we are only interested in the Southern Ocean. Write the code below to refine the index file (*argo_synthetic-profile_index_exercise1.csv* in Lesson 1) and download the relevant data.

In [7]:
df_index_ex1 = pd.read_csv('argo_synthetic-profile_index_ex1.csv',parse_dates=['datetime'])
df_index_so = df_index_ex1[df_index_ex1['latitude'] < -30]
download_argo(df_index_so, None)

⚠️ No specific floats provided.
⚠️ Defaulting to download ALL 27 files in the provided DataFrame!
🌍 Successfully connected to https://data-argo.ifremer.fr/
✅ Download successful: 1902647/profiles/SR1902647_046.nc
✅ Download successful: 1902746/profiles/SR1902746_003.nc
✅ Download successful: 2903440/profiles/SR2903440_050.nc
✅ Download successful: 2903864/profiles/SR2903864_070.nc
✅ Download successful: 2903865/profiles/SR2903865_070.nc
✅ Download successful: 2903969/profiles/SR2903969_010.nc
✅ Download successful: 4903566/profiles/SR4903566_002.nc
✅ Download successful: 4903857/profiles/SR4903857_007.nc
✅ Download successful: 5906204/profiles/SR5906204_220.nc
✅ Download successful: 5906311/profiles/SR5906311_188.nc
✅ Download successful: 5906554/profiles/SR5906554_113.nc
✅ Download successful: 7902124/profiles/SR7902124_047.nc
✅ Download successful: 7902147/profiles/SR7902147_038.nc
✅ Download successful: 7902163/profiles/SR7902163_005.nc
✅ Download successful: 7902165/profiles/SR7902

### Exercise 2: Download the full-sensor float time series after the testing phase.

From *timeline_ex2.png* in Lesson 2, we see that the float `5905531` was profiling more frequently (every few days) in the first month compared to the rest of the period. The former likely represents a testing phase for the float after deployment. We want to exclude this phase from analysis. Write the code below to refine the index (argo_synthetic-profile_index_exercise2.csv in Lesson 1) to exclude this phase and download the float time series.

In [9]:
df_index_ex2 = pd.read_csv('argo_synthetic-profile_index_exercise2.csv',parse_dates=['datetime'])
df_index_time = df_index_ex2[df_index_ex2['datetime'] > pd.to_datetime('2023-02-01 00:00:00')]
download_argo(df_index_time, '5905531')

🌍 Successfully connected to https://data-argo.ifremer.fr/
✅ Download successful: 5905531/5905531_Sprof.nc


### Exercise 3: Download the two sister floats that are slowly drifting near the Equator in the eastern Pacific

From Exercise 3 in Lesson 2, we found two floats (#20 and #21) that were operating near and at the same time and have the sequential WMO IDs. We are interested in how the float time series between these two floats compare. Write the code below to refine the index (argo_synthetic-profile_index_ex3.csv from Exercise 3 in Lesson 1) to download the two time series starting from January 1, 2022.

In [10]:
df_index = pd.read_csv('argo_synthetic-profile_index_ex3.csv',parse_dates=['datetime'])
df_index_2022 = df_index[df_index["datetime"] > pd.to_datetime("2022-01-01 00:00:00")]
download_argo(df_index_2022, [5906475, 5906476])

🌍 Successfully connected to https://data-argo.ifremer.fr/
✅ Download successful: 5906475/5906475_Sprof.nc
✅ Download successful: 5906476/5906476_Sprof.nc



---

This is the end of the lesson. If you are using **Binder**, don't forget to dowload the files created in this lesson before you lose connection!

Well done 🎉 Take a break 💤, have another cup ☕, and move to the next lesson ✍️ when you are ready 💪

While your memory is fresh, please feel free to provide your user experience on this lesson by visiting [this link](https://forms.gle/oAGmz5RTW4Pp46bt7). Thanks!